In [ ]:
!git clone -q https://github.com/Mikayla-ds2/surge-sense.git

In [ ]:
%cd surge-sense

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error

%pip install pmdarima
import pmdarima as pm

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/daily_visits.csv')

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data['admission_day'] = pd.to_datetime(data['admission_day'])

In [ ]:
plot = data.plot(x='admission_day', y='visit_count', legend=False, figsize=(20, 8))
plt.xticks(rotation=90, fontsize=10)
plt.title('# of Visits')
plt.xlabel('Time')
plt.ylabel('# of Visits')
plt.show()

In [ ]:
data['month_day'] = data['admission_day'].dt.strftime('%m-%d')
data.head()

In [ ]:
data['month'] = data['admission_day'].dt.strftime('%m')
data['day'] = data['admission_day'].dt.strftime('%d')
data['weekday'] = data['admission_day'].dt.day_name()
data.head()

In [ ]:
sum_visits = data.groupby('month_day', as_index=False)['visit_count'].sum()
agg_visits_sum = sum_visits.sort_values('month_day')

display(agg_visits_sum.loc[agg_visits_sum['visit_count'].nlargest(10).index])

plot = agg_visits_sum.plot(x='month_day', y='visit_count', legend=False, figsize=(20, 8))
plt.xticks(rotation=90, fontsize=10)
plt.title('Total Visits by Month and Day')
plt.xlabel('Month-Day')
plt.ylabel('Total Visit Count')
plt.show()

In [ ]:
avg_visits = data.groupby('month_day', as_index=False)['visit_count'].mean()
agg_visits_avg = avg_visits.sort_values('month_day')

display(agg_visits_avg.loc[agg_visits_avg['visit_count'].nlargest(10).index])

plot = agg_visits_avg.plot(x='month_day', y='visit_count', legend=False, figsize=(20, 8))
plt.xticks(rotation=90, fontsize=10)
plt.title('Average Visits by Month and Day')
plt.xlabel('Month-Day')
plt.ylabel('Average Visit Count')
plt.show()

In [ ]:
monthly_avg_visits = data.groupby('month', as_index=False)['visit_count'].mean()
agg_monthly_avg_visits = monthly_avg_visits.sort_values('month')

display(agg_monthly_avg_visits.loc[agg_monthly_avg_visits['visit_count'].nsmallest(12).index])

plot = agg_monthly_avg_visits.plot(x='month', y='visit_count', legend=False, figsize=(16, 6))
plt.xticks(rotation=90, fontsize=10)
plt.title('Average Visits by Month')
plt.xlabel('Month')
plt.ylabel('Average Visit Count')
plt.show()

In [ ]:
data['quarter'] = pd.PeriodIndex(data.admission_day, freq='Q')
data.head()

In [ ]:
quarterly_avg_visits = data.groupby('quarter', as_index=False)['visit_count'].mean()
agg_visits = quarterly_avg_visits.sort_values('quarter')

display(agg_visits.loc[agg_visits['visit_count'].nlargest(10).index])    

plot = agg_visits.plot(x='quarter', y='visit_count', legend=False, figsize=(12, 6))
plt.xticks(rotation=90, fontsize=10)
plt.title('Average Visits by Quarter')
plt.xlabel('Quarter')
plt.ylabel('Average Visit Count')
plt.show()

In [ ]:
data.visit_count.describe()

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(data['visit_count'], bins=15)
plt.show()

In [ ]:
# outlier detection - 
df = data.copy()
df['z_score'] = np.abs((df['visit_count'] - df['visit_count'].mean()) / df['visit_count'].std())
outliers_z_score = df[df['z_score'] > 2]
z_outlier_percent = (len(outliers_z_score) / len(df)) * 100

print(f"Detected {len(outliers_z_score)} outliers ({z_outlier_percent:.2f}% of the dataset) using z-scores; threshold = 2.\n")

ax = data.plot(x='admission_day', y='visit_count', label="Time Series", figsize=(20, 6))
outlier_dates = matplotlib.dates.date2num(outliers_z_score['admission_day'])
ax.scatter(outlier_dates, outliers_z_score['visit_count'], color='red', label='Detected outliers')
ax.legend()
plt.show()

In [ ]:
# outlier detection
df = data.copy()
df['z_score'] = np.abs((df['visit_count'] - df['visit_count'].mean()) / df['visit_count'].std())
outliers_z3_score = df[df['z_score'] > 3]
z_outlier_percent = (len(outliers_z3_score) / len(df)) * 100

print(f"Detected {len(outliers_z3_score)} outliers ({z_outlier_percent:.2f}% of the dataset) using z-scores; threshold = 3.\n")

ax = data.plot(x='admission_day', y='visit_count', label="Time Series", figsize=(20, 6))
outlier_dates = matplotlib.dates.date2num(outliers_z3_score['admission_day'])
ax.scatter(outlier_dates, outliers_z3_score['visit_count'], color='red', label='Detected outliers')
ax.legend()
plt.show()

In [ ]:
outlier_days = outliers_z3_score[['admission_day']].copy()
outlier_days['admission_day'] = outlier_days['admission_day'].dt.strftime('%Y-%m-%d')
display(outlier_days)

In [ ]:
outlier_days['admission_day'] = pd.to_datetime(outlier_days['admission_day'])
outlier_days['month_day'] = outlier_days['admission_day'].dt.strftime('%m-%d')
outlier_days['month'] = outlier_days['admission_day'].dt.strftime('%m')
outlier_days['day'] = outlier_days['admission_day'].dt.strftime('%d')
outlier_days['quarter'] = pd.PeriodIndex(outlier_days.admission_day, freq='Q')
outlier_days.head()